# PowerPlan Scheduler: Intelligent Outage-Aware Appliance Scheduling

## CSC-471-L Artificial Intelligence — Lab 6 (Open-Ended Lab)

**Institution:** Iqra University, Dept. of Computer Science (BS-CS)

**Student:** Dua Rehan | **Roll No:** 67238

**Instructor:** Aimen Zafar | **Section:** Wednesday 2:30–5:20

**Evaluation:** Bloom's Taxonomy C3 (Application), P2 (Mechanism) | Graduate Attributes GA-2 (Problem Analysis), GA-5 (Modern Tool Usage)

## 1. Problem Formulation & Background Theory

### The Appliance Scheduling Problem as State-Space Search

We model **household appliance scheduling under power outages and fluctuating tariffs** as a **deterministic, fully observable state-space search problem**. A fixed set of $N$ household appliances must be allocated across $M$ discrete time slots, subject to:

- **Grid capacity constraints:** The total power draw of all concurrently running appliances in any slot must not exceed the slot's capacity limit $C_t$.
- **Scheduled power outages:** Certain slots are marked as outage periods where **no** appliance may operate (forced IDLE).
- **Fluctuating energy tariffs:** Each active slot has a per-unit energy price $p_t$, motivating cost-aware scheduling.

### State-Space Definition

Each state $s$ in the search tree is uniquely identified by:

$$S = (t, (r_1, r_2, \ldots, r_N))$$

where $t \in \{0, 1, \ldots, M\}$ is the current time slot index and $r_i \in \{0, 1, \ldots, D_i\}$ is the remaining number of slots needed for appliance $i$. The canonical graph-search key is:

$$\text{key}(s) = (t, (r_1, r_2, \ldots, r_N))$$

Alongside the core state key, `ScheduleState` stores path metadata: `assignments` (tuple of job-ID tuples per slot), `g_cost` (cumulative energy cost), and `completion_slot` (1-based index of last active slot).

### Action Generation with Outage Enforcement & Capacity Constraints

At state $s$ with current slot index $t < M$ and unfinished jobs $U(s) = \{i \mid r_i > 0\}$:

1. **Goal Check:** If $U(s) = \emptyset$, all jobs are complete — no further actions needed.
2. **Horizon Exhausted:** If $t = M$ and $U(s) \neq \emptyset$, the branch is a dead end.
3. **Outage Check:** If `slots[t].is_outage == True`, the only legal action is `IDLE` = `()`.
4. **Active Slot:** Generate all subsets $A \subseteq U(s)$ satisfying $\sum_{i \in A} P_i \leq C_t$, including $A = \emptyset$ (voluntary IDLE) so A* can skip expensive slots.

### Three Search Strategies

| Strategy | Data Structure | Optimization Target |
|----------|---------------|--------------------|
| **BFS** | FIFO Queue (`collections.deque`) | Earliest completion time (shallowest goal slot) |
| **DFS** | LIFO Stack (`list`) | First feasible schedule with minimal memory footprint |
| **A\*** | Priority Queue (`heapq`) with $f(n) = g(n) + h(n)$ | Minimum total energy cost (globally optimal) |

### Admissible Heuristic for A*

The A* heuristic is defined as:

$$h(n) = E_{\text{rem}}(s) \times p_{\min}(t)$$

where $E_{\text{rem}}(s) = \sum_{i=1}^{N} r_i \times P_i$ is the total remaining energy requirement, and $p_{\min}(t) = \min_{k \in \{t,\ldots,M-1\},\ \text{outage}_k = \text{False}} p_k$ is the cheapest future active tariff.

**Proof of Admissibility:** Every remaining energy unit in $E_{\text{rem}}(s)$ must be scheduled in some active slot $k \geq t$ at a tariff $p_k \geq p_{\min}(t)$. By relaxing the grid capacity constraint and assuming all remaining energy can be purchased at the single lowest future tariff $p_{\min}(t)$, we guarantee $h(s) \leq h^*(s)$ — the heuristic never overestimates the true minimum remaining cost. Therefore, A* with this heuristic is guaranteed to find the globally cost-optimal schedule.

In [ ]:
%matplotlib inline

from __future__ import annotations
import sys
import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

# ---------------------------------------------------------------------------
# Cell 3: ApplianceJob & PowerSlot Dataclass Definitions
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class ApplianceJob:
    """Represents a household appliance task requiring power allocation."""
    job_id: str
    name: str
    power_units: int          # Power drawn per active slot (e.g., kW or abstract units)
    duration_slots: int = 1   # Number of slots required to finish the job
    priority: int = 1         # Optional tie-breaking priority (1 = highest)

    def __post_init__(self) -> None:
        if self.power_units <= 0:
            raise ValueError(f"Appliance {self.name} power_units must be positive.")
        if self.duration_slots <= 0:
            raise ValueError(f"Appliance {self.name} duration_slots must be positive.")

    @property
    def total_energy_units(self) -> int:
        """Total energy units required across the entire duration of the job."""
        return self.power_units * self.duration_slots


@dataclass(frozen=True)
class PowerSlot:
    """Represents a discrete chronological time block in the power grid schedule."""
    slot_index: int           # 1-based chronological slot number (1..M)
    label: str                # Human-readable slot description
    is_outage: bool           # True if scheduled load-shedding / outage
    capacity_limit: int       # Maximum simultaneous power units allowed in this slot
    unit_price: float         # Tariff per energy unit in this slot ($ or PKR)

    @property
    def effective_capacity(self) -> int:
        """Returns 0 during an outage, otherwise capacity_limit."""
        return 0 if self.is_outage else self.capacity_limit


print("\u2705 ApplianceJob and PowerSlot dataclasses defined successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4: ScheduleState & SearchResult Dataclass Definitions
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class ScheduleState:
    """
    Immutable state representation in the scheduling state space.
    - current_slot_index: 0-based index into the environment's slots list.
    - remaining_durations: tuple of remaining slot counts for each job (aligned with env.jobs).
    - assignments: tuple of tuples, where assignments[t] holds job_ids scheduled in slot t.
    - g_cost: cumulative energy cost incurred up to current_slot_index.
    - completion_slot: 1-based slot number where the final appliance completed (0 if incomplete).
    """
    current_slot_index: int
    remaining_durations: Tuple[int, ...]
    assignments: Tuple[Tuple[str, ...], ...] = field(default_factory=tuple)
    g_cost: float = 0.0
    completion_slot: int = 0

    @property
    def state_key(self) -> Tuple[int, Tuple[int, ...]]:
        """Canonical graph-search key identifying the Markovian state."""
        return (self.current_slot_index, self.remaining_durations)

    def is_all_completed(self) -> bool:
        """Returns True when every appliance has 0 remaining duration."""
        return all(rem == 0 for rem in self.remaining_durations)


@dataclass
class SearchResult:
    """Encapsulates the solution schedule and telemetry metrics for a search run."""
    algorithm_name: str
    goal_Description: str
    success: bool
    final_state: Optional[ScheduleState]
    total_cost: float
    completion_slot: int
    nodes_expanded: int
    nodes_generated: int
    max_frontier_size: int
    execution_time_ms: float
    slot_assignments: Dict[int, List[str]]          # 1-based slot_index -> list of job_ids
    slot_loads: Dict[int, int]                      # 1-based slot_index -> total power units used
    slot_costs: Dict[int, float]                    # 1-based slot_index -> cost incurred in slot
    search_trace: List[Dict[str, object]] = field(default_factory=list)


print("\u2705 ScheduleState and SearchResult dataclasses defined successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5: SchedulingEnvironment Class (State-Action Transition Model)
# ---------------------------------------------------------------------------

from itertools import combinations


class SchedulingEnvironment:
    """
    Deterministic, fully observable state-action transition environment for PowerPlan.
    Models outage enforcement, grid capacity filtering, cost accumulation, and heuristic bounds.
    """

    def __init__(self, jobs: List[ApplianceJob], slots: List[PowerSlot], name: str = "Default") -> None:
        if not jobs:
            raise ValueError("SchedulingEnvironment requires at least one ApplianceJob.")
        if not slots:
            raise ValueError("SchedulingEnvironment requires at least one PowerSlot.")
        self.name = name
        self.jobs: Tuple[ApplianceJob, ...] = tuple(jobs)
        self.slots: Tuple[PowerSlot, ...] = tuple(slots)
        self.job_map: Dict[str, ApplianceJob] = {job.job_id: job for job in self.jobs}
        self.job_index_map: Dict[str, int] = {job.job_id: idx for idx, job in enumerate(self.jobs)}

    def get_initial_state(self) -> ScheduleState:
        """Constructs the root ScheduleState at slot index 0 with full job durations."""
        initial_remaining = tuple(job.duration_slots for job in self.jobs)
        return ScheduleState(
            current_slot_index=0,
            remaining_durations=initial_remaining,
            assignments=tuple(),
            g_cost=0.0,
            completion_slot=0,
        )

    def is_goal_state(self, state: ScheduleState) -> bool:
        """Goal test: returns True as soon as all jobs have 0 remaining duration."""
        return state.is_all_completed()

    def get_remaining_energy(self, state: ScheduleState) -> int:
        """Calculates total remaining energy units required across all unfinished jobs."""
        return sum(
            rem * self.jobs[idx].power_units
            for idx, rem in enumerate(state.remaining_durations)
        )

    def is_remaining_feasible(self, state: ScheduleState) -> bool:
        """
        Lookahead feasibility check: verifies whether remaining active slots have enough
        aggregate capacity and active slot count to finish all remaining work.
        """
        if state.is_all_completed():
            return True
        if state.current_slot_index >= len(self.slots):
            return False

        remaining_slots = self.slots[state.current_slot_index:]
        active_slots = [s for s in remaining_slots if not s.is_outage]

        max_needed_slots = max(state.remaining_durations)
        if max_needed_slots > len(active_slots):
            return False

        total_remaining_capacity = sum(s.effective_capacity for s in active_slots)
        if self.get_remaining_energy(state) > total_remaining_capacity:
            return False

        return True

    def get_valid_actions(self, state: ScheduleState) -> List[Tuple[str, ...]]:
        """
        Generates all valid appliance combinations for the current slot:
        - If all jobs are already completed or horizon is reached, returns [].
        - If current slot is an OUTAGE, forces IDLE: returns [()].
        - Otherwise, generates all subsets of unfinished jobs whose combined power_units
          do not exceed slot.capacity_limit, including () (IDLE).
        """
        if state.is_all_completed() or state.current_slot_index >= len(self.slots):
            return []

        current_slot = self.slots[state.current_slot_index]

        # Flowchart Rule: Is Current Slot an Outage? -> Yes -> Action: Force IDLE
        if current_slot.is_outage:
            return [tuple()]

        # Identify unfinished job indices
        unfinished_indices = [
            idx for idx, rem in enumerate(state.remaining_durations) if rem > 0
        ]

        valid_actions: List[Tuple[str, ...]] = [tuple()]  # Always include IDLE action ()
        capacity = current_slot.capacity_limit

        for r in range(1, len(unfinished_indices) + 1):
            for combo_indices in combinations(unfinished_indices, r):
                combo_power = sum(self.jobs[i].power_units for i in combo_indices)
                if combo_power <= capacity:
                    action = tuple(self.jobs[i].job_id for i in combo_indices)
                    valid_actions.append(action)

        return valid_actions

    def apply_action(self, state: ScheduleState, action: Tuple[str, ...]) -> ScheduleState:
        """
        Applies a valid action (tuple of job_ids) in the current slot and returns the next ScheduleState.
        """
        slot = self.slots[state.current_slot_index]
        if slot.is_outage and len(action) > 0:
            raise ValueError(f"Cannot schedule jobs {action} during outage slot {slot.slot_index}.")

        action_power = sum(self.job_map[jid].power_units for jid in action)
        if action_power > slot.effective_capacity:
            raise ValueError(
                f"Action {action} power ({action_power}u) exceeds slot {slot.slot_index} "
                f"capacity ({slot.effective_capacity}u)."
            )

        new_remaining = list(state.remaining_durations)
        for jid in action:
            idx = self.job_index_map[jid]
            if new_remaining[idx] <= 0:
                raise ValueError(f"Job {jid} is already completed.")
            new_remaining[idx] -= 1

        step_cost = action_power * slot.unit_price
        next_slot_number = slot.slot_index  # 1-based slot number
        new_completion_slot = (
            next_slot_number if len(action) > 0 else state.completion_slot
        )

        return ScheduleState(
            current_slot_index=state.current_slot_index + 1,
            remaining_durations=tuple(new_remaining),
            assignments=state.assignments + (action,),
            g_cost=round(state.g_cost + step_cost, 6),
            completion_slot=new_completion_slot,
        )

    def calculate_heuristic(self, state: ScheduleState) -> float:
        """
        Admissible A* heuristic:
        h(n) = Remaining Energy Units * Lowest Future Active Slot Unit Price.
        Returns float('inf') if remaining jobs cannot possibly be completed in future slots.
        """
        if state.is_all_completed():
            return 0.0

        if not self.is_remaining_feasible(state):
            return float("inf")

        future_active_prices = [
            s.unit_price
            for s in self.slots[state.current_slot_index:]
            if not s.is_outage
        ]
        if not future_active_prices:
            return float("inf")

        min_future_price = min(future_active_prices)
        remaining_energy = self.get_remaining_energy(state)
        return round(remaining_energy * min_future_price, 6)


print("\u2705 SchedulingEnvironment class defined successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6: SchedulingAgent Class (BFS, DFS, A* Search Implementations)
# ---------------------------------------------------------------------------

from collections import deque
import heapq
import itertools
import time


class SchedulingAgent:
    """
    Intelligent scheduling agent implementing BFS, DFS, and A* search strategies
    over the SchedulingEnvironment state space.
    """

    def __init__(self, env: SchedulingEnvironment) -> None:
        self.env = env

    def _build_result(
        self,
        algorithm_name: str,
        goal_description: str,
        final_state: Optional[ScheduleState],
        nodes_expanded: int,
        nodes_generated: int,
        max_frontier_size: int,
        start_time: float,
        search_trace: List[Dict[str, object]],
    ) -> SearchResult:
        elapsed_ms = round((time.perf_counter() - start_time) * 1000.0, 3)
        slot_assignments: Dict[int, List[str]] = {}
        slot_loads: Dict[int, int] = {}
        slot_costs: Dict[int, float] = {}

        for slot in self.env.slots:
            s_idx = slot.slot_index
            if final_state is not None and (s_idx - 1) < len(final_state.assignments):
                assigned = list(final_state.assignments[s_idx - 1])
            else:
                assigned = []
            load = sum(self.env.job_map[jid].power_units for jid in assigned)
            cost = round(load * slot.unit_price, 2)
            slot_assignments[s_idx] = assigned
            slot_loads[s_idx] = load
            slot_costs[s_idx] = cost

        return SearchResult(
            algorithm_name=algorithm_name,
            goal_Description=goal_description,
            success=final_state is not None,
            final_state=final_state,
            total_cost=round(final_state.g_cost, 2) if final_state else float("inf"),
            completion_slot=final_state.completion_slot if final_state else -1,
            nodes_expanded=nodes_expanded,
            nodes_generated=nodes_generated,
            max_frontier_size=max_frontier_size,
            execution_time_ms=elapsed_ms,
            slot_assignments=slot_assignments,
            slot_loads=slot_loads,
            slot_costs=slot_costs,
            search_trace=search_trace,
        )

    def solve_bfs(self) -> SearchResult:
        """
        Breadth-First Search (FIFO Queue).
        Optimizes for Earliest Completion Time (shallowest slot depth where all jobs finish).
        """
        start_time = time.perf_counter()
        initial_state = self.env.get_initial_state()
        if self.env.is_goal_state(initial_state):
            return self._build_result(
                "BFS (Breadth-First Search)",
                "Earliest Completion Time",
                initial_state, 0, 1, 1, start_time, []
            )

        frontier: deque = deque([initial_state])
        visited = {initial_state.state_key}
        nodes_expanded = 0
        nodes_generated = 1
        max_frontier_size = 1
        trace: List[Dict[str, object]] = []

        while frontier:
            max_frontier_size = max(max_frontier_size, len(frontier))
            state = frontier.popleft()
            nodes_expanded += 1

            # Sort actions by descending total energy so BFS prioritizes earliest completion
            actions = self.env.get_valid_actions(state)
            actions.sort(
                key=lambda act: (
                    sum(self.env.job_map[jid].power_units for jid in act),
                    len(act),
                ),
                reverse=True,
            )

            for action in actions:
                next_state = self.env.apply_action(state, action)
                nodes_generated += 1

                if not self.env.is_remaining_feasible(next_state):
                    continue

                if self.env.is_goal_state(next_state):
                    trace.append({
                        "step": nodes_expanded,
                        "slot": state.current_slot_index + 1,
                        "action": action,
                        "g_cost": next_state.g_cost,
                        "event": "GOAL_FOUND",
                    })
                    return self._build_result(
                        "BFS (Breadth-First Search)",
                        "Earliest Completion Time",
                        next_state,
                        nodes_expanded,
                        nodes_generated,
                        max_frontier_size,
                        start_time,
                        trace,
                    )

                if next_state.state_key not in visited:
                    visited.add(next_state.state_key)
                    frontier.append(next_state)

        return self._build_result(
            "BFS (Breadth-First Search)",
            "Earliest Completion Time",
            None, nodes_expanded, nodes_generated, max_frontier_size, start_time, trace,
        )

    def solve_dfs(self) -> SearchResult:
        """
        Depth-First Search (LIFO Stack).
        Finds the first computationally feasible schedule with minimal memory overhead.
        """
        start_time = time.perf_counter()
        initial_state = self.env.get_initial_state()
        stack: List[ScheduleState] = [initial_state]
        visited = set()
        nodes_expanded = 0
        nodes_generated = 1
        max_frontier_size = 1
        trace: List[Dict[str, object]] = []

        while stack:
            max_frontier_size = max(max_frontier_size, len(stack))
            state = stack.pop()

            if self.env.is_goal_state(state):
                return self._build_result(
                    "DFS (Depth-First Search)",
                    "First Feasible Schedule (Low Memory)",
                    state,
                    nodes_expanded,
                    nodes_generated,
                    max_frontier_size,
                    start_time,
                    trace,
                )

            if state.state_key in visited:
                continue
            visited.add(state.state_key)
            nodes_expanded += 1

            actions = self.env.get_valid_actions(state)
            # Push IDLE first; moderate single-appliance actions pop first from LIFO
            actions.sort(
                key=lambda act: (
                    len(act) == 1,
                    len(act) > 1,
                    -sum(self.env.job_map[jid].power_units for jid in act),
                )
            )

            for action in actions:
                next_state = self.env.apply_action(state, action)
                nodes_generated += 1
                if (
                    next_state.state_key not in visited
                    and self.env.is_remaining_feasible(next_state)
                ):
                    stack.append(next_state)

        return self._build_result(
            "DFS (Depth-First Search)",
            "First Feasible Schedule (Low Memory)",
            None, nodes_expanded, nodes_generated, max_frontier_size, start_time, trace,
        )

    def solve_astar(self) -> SearchResult:
        """
        A* Search (Priority Queue ordered by f(n) = g(n) + h(n)).
        Uses the admissible heuristic h(n) = Remaining Energy * Lowest Future Active Price
        to guarantee the minimum total energy cost schedule.
        """
        start_time = time.perf_counter()
        initial_state = self.env.get_initial_state()
        counter = itertools.count()

        init_h = self.env.calculate_heuristic(initial_state)
        # Priority tuple: (f_cost, h_cost, completion_slot, tie_breaker, state)
        frontier = [
            (init_h, init_h, 0, next(counter), initial_state)
        ]
        best_g = {initial_state.state_key: 0.0}

        nodes_expanded = 0
        nodes_generated = 1
        max_frontier_size = 1
        trace: List[Dict[str, object]] = []

        while frontier:
            max_frontier_size = max(max_frontier_size, len(frontier))
            f_cost, h_cost, _, _, state = heapq.heappop(frontier)

            if self.env.is_goal_state(state):
                return self._build_result(
                    "A* Search (Heuristic Cost-Optimal)",
                    "Minimum Total Energy Cost",
                    state,
                    nodes_expanded,
                    nodes_generated,
                    max_frontier_size,
                    start_time,
                    trace,
                )

            # Skip stale priority queue entries
            if state.g_cost > best_g.get(state.state_key, float("inf")):
                continue

            nodes_expanded += 1

            for action in self.env.get_valid_actions(state):
                next_state = self.env.apply_action(state, action)
                nodes_generated += 1

                h_next = self.env.calculate_heuristic(next_state)
                if h_next == float("inf"):
                    continue  # Pruned by feasibility/admissible heuristic bound

                g_next = next_state.g_cost
                f_next = round(g_next + h_next, 6)

                # Goal states compete strictly on g_cost
                canonical_key = (
                    (-1, next_state.remaining_durations)
                    if next_state.is_all_completed()
                    else next_state.state_key
                )

                if g_next < best_g.get(canonical_key, float("inf")):
                    best_g[canonical_key] = g_next
                    heapq.heappush(
                        frontier,
                        (f_next, h_next, next_state.completion_slot, next(counter), next_state),
                    )

        return self._build_result(
            "A* Search (Heuristic Cost-Optimal)",
            "Minimum Total Energy Cost",
            None, nodes_expanded, nodes_generated, max_frontier_size, start_time, trace,
        )

    def run_all_comparisons(self) -> Dict[str, SearchResult]:
        """Executes BFS, DFS, and A* on the same initialized environment and returns results."""
        return {
            "BFS": self.solve_bfs(),
            "DFS": self.solve_dfs(),
            "A*": self.solve_astar(),
        }


print("\u2705 SchedulingAgent class (BFS, DFS, A*) defined successfully.")

## 2. Scenario 1: Proposal 6-Slot Benchmark

The default calibrated scenario uses **4 appliances** (Water Pump 2u, Washing Machine 3u, Dishwasher 2u, EV Charger 4u) across **6 time slots** with a grid capacity of 6 units/slot. Slot 2 is a scheduled outage. Tariffs range from \$4/u to \$20/u, carefully calibrated so that BFS, DFS, and A* produce **distinctly different** schedules.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 8: Scenario 1 — Create Environment, Run All Algorithms, Print Summary
# ---------------------------------------------------------------------------

# Create Scenario 1: Proposal 6-Slot Benchmark
scenario1_jobs = [
    ApplianceJob(job_id="WP", name="Water Pump", power_units=2, duration_slots=1, priority=1),
    ApplianceJob(job_id="WM", name="Washing Machine", power_units=3, duration_slots=1, priority=2),
    ApplianceJob(job_id="DW", name="Dishwasher", power_units=2, duration_slots=1, priority=3),
    ApplianceJob(job_id="EV", name="EV Charger", power_units=4, duration_slots=1, priority=1),
]

scenario1_slots = [
    PowerSlot(slot_index=1, label="Slot 1 (Morning)",     is_outage=False, capacity_limit=6, unit_price=10.0),
    PowerSlot(slot_index=2, label="Slot 2 (Outage)",      is_outage=True,  capacity_limit=6, unit_price=0.0),
    PowerSlot(slot_index=3, label="Slot 3 (Peak)",        is_outage=False, capacity_limit=6, unit_price=15.0),
    PowerSlot(slot_index=4, label="Slot 4 (Shoulder)",    is_outage=False, capacity_limit=6, unit_price=8.0),
    PowerSlot(slot_index=5, label="Slot 5 (Super Peak)",  is_outage=False, capacity_limit=6, unit_price=20.0),
    PowerSlot(slot_index=6, label="Slot 6 (Off-Peak)",    is_outage=False, capacity_limit=6, unit_price=4.0),
]

env1 = SchedulingEnvironment(jobs=scenario1_jobs, slots=scenario1_slots, name="Proposal_6Slot_Benchmark")
agent1 = SchedulingAgent(env1)
results1 = agent1.run_all_comparisons()

# Print comparative summary table
print("=" * 90)
print(f" POWERPLAN SCHEDULER — AI STATE-SPACE SEARCH BENCHMARK ({env1.name})")
print("=" * 90)
header = (
    f"{'Algorithm':<34} | {'Cost ($)':>9} | {'Done Slot':>9} | "
    f"{'Expanded':>8} | {'Max Queue':>9} | {'Time (ms)':>9}"
)
print(header)
print("-" * 90)
for key in ["BFS", "DFS", "A*"]:
    r = results1[key]
    print(
        f"{r.algorithm_name:<34} | {r.total_cost:>9.2f} | "
        f"{'Slot ' + str(r.completion_slot):>9} | {r.nodes_expanded:>8d} | "
        f"{r.max_frontier_size:>9d} | {r.execution_time_ms:>9.3f}"
    )
print("=" * 90)

# Print detailed slot-by-slot schedules
for key in ["BFS", "DFS", "A*"]:
    r = results1[key]
    print(f"\n[{key}] Detailed Slot Schedule ({r.goal_Description}):")
    for slot in env1.slots:
        s_idx = slot.slot_index
        jobs_str = ", ".join(r.slot_assignments[s_idx]) if r.slot_assignments[s_idx] else "IDLE"
        status_tag = "OUTAGE (FORCED IDLE)" if slot.is_outage else f"Active (${slot.unit_price:.0f}/u)"
        print(
            f"  - Slot {s_idx} [{status_tag:<20}]: {jobs_str:<18} "
            f"| Load: {r.slot_loads[s_idx]}/{slot.effective_capacity}u "
            f"| Slot Cost: ${r.slot_costs[s_idx]:.2f}"
        )

### Schedule Heatmap — BFS vs DFS vs A*

In [ ]:
# ---------------------------------------------------------------------------
# Cell 10: Render Schedule Heatmap — 3-Panel Side-by-Side Comparison
# ---------------------------------------------------------------------------

import matplotlib.pyplot as plt
import numpy as np


def render_schedule_heatmap(env, results, title_suffix=""):
    """Renders a 3-panel side-by-side Schedule Heatmap comparing BFS, DFS, and A*."""
    algo_keys = ["BFS", "DFS", "A*"]
    num_jobs = len(env.jobs)
    num_slots = len(env.slots)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6.2), sharey=True)
    fig.suptitle(
        f"PowerPlan Schedule Heatmap Comparison — {env.name}{title_suffix}\n"
        "(Cell Value = Power Units Allocated | Red Column = Outage Slot)",
        fontsize=14, fontweight="bold", y=0.98,
    )

    job_labels = [f"{j.name}\n({j.job_id}: {j.power_units}u)" for j in env.jobs]
    slot_labels = [
        f"S{s.slot_index}\n{'OUTAGE' if s.is_outage else f'${s.unit_price:.0f}/u'}"
        for s in env.slots
    ]

    for ax_idx, algo_key in enumerate(algo_keys):
        ax = axes[ax_idx]
        res = results[algo_key]
        matrix = np.zeros((num_jobs, num_slots), dtype=float)

        for j_idx, job in enumerate(env.jobs):
            for s_idx, slot in enumerate(env.slots):
                if slot.is_outage:
                    matrix[j_idx, s_idx] = -1.0
                elif job.job_id in res.slot_assignments.get(slot.slot_index, []):
                    matrix[j_idx, s_idx] = float(job.power_units)

        display_rgb = np.zeros((num_jobs, num_slots, 3), dtype=float)
        for r in range(num_jobs):
            for c in range(num_slots):
                val = matrix[r, c]
                if val < 0:
                    display_rgb[r, c] = (0.96, 0.82, 0.82)
                elif val == 0:
                    display_rgb[r, c] = (0.95, 0.96, 0.98)
                else:
                    intensity = min(val / 4.5, 1.0)
                    display_rgb[r, c] = (
                        0.12 * (1 - intensity) + 0.05,
                        0.55 * (1 - 0.3 * intensity) + 0.15,
                        0.68 * (1 - 0.2 * intensity) + 0.20,
                    )

        ax.imshow(display_rgb, aspect="auto")

        for r in range(num_jobs):
            for c in range(num_slots):
                val = matrix[r, c]
                if val < 0:
                    ax.text(c, r, "OUTAGE\n(IDLE)", ha="center", va="center",
                            fontsize=8.5, color="#8B1E1E", fontweight="bold")
                elif val == 0:
                    ax.text(c, r, "\u2014", ha="center", va="center",
                            fontsize=11, color="#7A869A")
                else:
                    ax.text(c, r, f"ON\n({int(val)}u)", ha="center", va="center",
                            fontsize=9.5, color="white", fontweight="bold")

        ax.set_xticks(np.arange(num_slots))
        ax.set_xticklabels(slot_labels, fontsize=9.5, fontweight="bold")
        ax.set_yticks(np.arange(num_jobs))
        if ax_idx == 0:
            ax.set_yticklabels(job_labels, fontsize=10, fontweight="bold")

        for s_idx, slot in enumerate(env.slots):
            load = res.slot_loads.get(slot.slot_index, 0)
            cap = slot.effective_capacity
            ax.text(
                s_idx, num_jobs - 0.52, f"Load: {load}/{cap}u",
                ha="center", va="bottom", fontsize=8,
                color="#1E293B" if not slot.is_outage else "#8B1E1E",
                bbox=dict(boxstyle="round,pad=0.18", facecolor="white", alpha=0.85, edgecolor="none"),
            )

        ax.set_title(
            f"{algo_key}: {res.goal_Description}\n"
            f"Total Cost: ${res.total_cost:.2f} | Completed: Slot {res.completion_slot} | "
            f"Expanded: {res.nodes_expanded} nodes",
            fontsize=10.5, fontweight="bold", pad=10,
        )

    plt.tight_layout(rect=[0, 0, 1, 0.91])
    plt.show()


render_schedule_heatmap(env1, results1)

### Algorithm Metrics Comparison

In [ ]:
# ---------------------------------------------------------------------------
# Cell 12: Render Algorithm Metrics Comparison — 4-Panel Bar Chart
# ---------------------------------------------------------------------------


def render_metrics_comparison(results, title_prefix=""):
    """Renders a 4-panel bar chart comparing key performance metrics across BFS, DFS, and A*."""
    algos = ["BFS", "DFS", "A*"]
    colors = ["#2563EB", "#D97706", "#059669"]

    metrics = [
        ("Total Energy Cost ($)", [results[k].total_cost for k in algos], "${:.1f}"),
        ("Completion Time (Slot #)", [results[k].completion_slot for k in algos], "Slot {:.0f}"),
        ("Nodes Expanded (Count)", [results[k].nodes_expanded for k in algos], "{:.0f}"),
        ("Peak Frontier Size (Nodes)", [results[k].max_frontier_size for k in algos], "{:.0f}"),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
    fig.suptitle(
        f"{title_prefix}PowerPlan Search Strategy Performance Trade-Offs",
        fontsize=14, fontweight="bold", y=1.02,
    )

    for idx, (title, values, fmt) in enumerate(metrics):
        ax = axes[idx]
        bars = ax.bar(algos, values, color=colors, width=0.55, edgecolor="#1E293B", linewidth=1.0)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        max_val = max(values) if max(values) > 0 else 1
        ax.set_ylim(0, max_val * 1.22)
        for bar, val in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                bar.get_height() + max_val * 0.03,
                fmt.format(val),
                ha="center", va="bottom", fontsize=9.5, fontweight="bold",
            )

    plt.tight_layout()
    plt.show()


render_metrics_comparison(results1)

## 3. Scenario 2: Karachi K-Electric Load-Shedding (8-Slot Multi-Duration)

This extended scenario models a realistic **Karachi K-Electric Time-of-Use (ToU) tariff** grid with **8 time slots** (2-hour blocks), **two scheduled outages**, and **multi-slot appliances** (the Dishwasher and EV Charger each require 2 consecutive slots of operation). This demonstrates the system's ability to handle more complex scheduling constraints.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 14: Scenario 2 — Karachi K-Electric Load-Shedding (8-Slot Multi-Duration)
# ---------------------------------------------------------------------------

scenario2_jobs = [
    ApplianceJob(job_id="WP", name="Water Pump", power_units=2, duration_slots=1),
    ApplianceJob(job_id="WM", name="Washing Machine", power_units=3, duration_slots=1),
    ApplianceJob(job_id="DW", name="Dishwasher", power_units=2, duration_slots=2),
    ApplianceJob(job_id="EV", name="EV Charger", power_units=4, duration_slots=2),
]

scenario2_slots = [
    PowerSlot(slot_index=1, label="08:00-10:00",             is_outage=False, capacity_limit=6, unit_price=12.0),
    PowerSlot(slot_index=2, label="10:00-12:00 (Outage)",    is_outage=True,  capacity_limit=6, unit_price=0.0),
    PowerSlot(slot_index=3, label="12:00-14:00",             is_outage=False, capacity_limit=6, unit_price=14.0),
    PowerSlot(slot_index=4, label="14:00-16:00",             is_outage=False, capacity_limit=6, unit_price=9.0),
    PowerSlot(slot_index=5, label="16:00-18:00 (Outage)",    is_outage=True,  capacity_limit=6, unit_price=0.0),
    PowerSlot(slot_index=6, label="18:00-20:00 (Peak)",      is_outage=False, capacity_limit=6, unit_price=22.0),
    PowerSlot(slot_index=7, label="20:00-22:00 (Peak)",      is_outage=False, capacity_limit=6, unit_price=18.0),
    PowerSlot(slot_index=8, label="22:00-00:00 (Off-Peak)",  is_outage=False, capacity_limit=6, unit_price=5.0),
]

env2 = SchedulingEnvironment(jobs=scenario2_jobs, slots=scenario2_slots, name="Karachi_MultiSlot_ToU")
agent2 = SchedulingAgent(env2)
results2 = agent2.run_all_comparisons()

# Print summary
print("=" * 90)
print(f" POWERPLAN SCHEDULER — AI STATE-SPACE SEARCH BENCHMARK ({env2.name})")
print("=" * 90)
header = (
    f"{'Algorithm':<34} | {'Cost ($)':>9} | {'Done Slot':>9} | "
    f"{'Expanded':>8} | {'Max Queue':>9} | {'Time (ms)':>9}"
)
print(header)
print("-" * 90)
for key in ["BFS", "DFS", "A*"]:
    r = results2[key]
    print(
        f"{r.algorithm_name:<34} | {r.total_cost:>9.2f} | "
        f"{'Slot ' + str(r.completion_slot):>9} | {r.nodes_expanded:>8d} | "
        f"{r.max_frontier_size:>9d} | {r.execution_time_ms:>9.3f}"
    )
print("=" * 90)

# Render heatmap for Scenario 2
render_schedule_heatmap(env2, results2)

## 4. Scenario 3: Combinatorial Scaling Analysis (3–7 Appliances)

This experiment benchmarks how the **state-space complexity** grows as the number of appliances increases from 3 to 7. We observe the **combinatorial explosion** of uninformed search (BFS/DFS) versus the **heuristic pruning** efficiency of A*, which dramatically reduces nodes expanded.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 16: Scenario 3 — Combinatorial Scaling Analysis (3 to 7 Appliances)
# ---------------------------------------------------------------------------


def create_scaling_environment(num_appliances: int) -> SchedulingEnvironment:
    """Creates a scaling test environment with num_appliances jobs across 6 slots."""
    catalog = [
        ("WP", "Water Pump", 2),
        ("WM", "Washing Machine", 3),
        ("DW", "Dishwasher", 2),
        ("EV", "EV Charger", 4),
        ("IH", "Induction Heater", 2),
        ("DR", "Clothes Dryer", 3),
        ("PP", "Pool Filtration Pump", 2),
    ]
    selected = catalog[:num_appliances]
    jobs = [ApplianceJob(job_id=jid, name=name, power_units=pwr, duration_slots=1) for jid, name, pwr in selected]
    slots = [
        PowerSlot(slot_index=1, label="Slot 1",           is_outage=False, capacity_limit=7, unit_price=10.0),
        PowerSlot(slot_index=2, label="Slot 2 (Outage)",  is_outage=True,  capacity_limit=7, unit_price=0.0),
        PowerSlot(slot_index=3, label="Slot 3",           is_outage=False, capacity_limit=7, unit_price=15.0),
        PowerSlot(slot_index=4, label="Slot 4",           is_outage=False, capacity_limit=7, unit_price=8.0),
        PowerSlot(slot_index=5, label="Slot 5",           is_outage=False, capacity_limit=7, unit_price=20.0),
        PowerSlot(slot_index=6, label="Slot 6",           is_outage=False, capacity_limit=7, unit_price=4.0),
    ]
    return SchedulingEnvironment(jobs=jobs, slots=slots, name=f"Scaling_{num_appliances}_Appliances")


# Run scaling benchmark
scaling_rows = []
print("\n" + "=" * 80)
print(" COMBINATORIAL SCALING ANALYSIS (3 to 7 Appliances)")
print("=" * 80)
print(f"{'N':>3} | {'BFS Expanded':>13} | {'DFS Expanded':>13} | {'A* Expanded':>13} | {'A* Cost ($)':>11}")
print("-" * 80)

for n in range(3, 8):
    env_s = create_scaling_environment(n)
    agent_s = SchedulingAgent(env_s)
    res_s = agent_s.run_all_comparisons()
    row = {
        "num_appliances": n,
        "bfs_expanded": res_s["BFS"].nodes_expanded,
        "dfs_expanded": res_s["DFS"].nodes_expanded,
        "astar_expanded": res_s["A*"].nodes_expanded,
        "astar_cost": res_s["A*"].total_cost,
    }
    scaling_rows.append(row)
    print(
        f"{n:>3} | {row['bfs_expanded']:>13,} | {row['dfs_expanded']:>13,} | "
        f"{row['astar_expanded']:>13,} | ${row['astar_cost']:>10.2f}"
    )

print("=" * 80)

# Render scaling plot
counts = [r["num_appliances"] for r in scaling_rows]
bfs_nodes = [r["bfs_expanded"] for r in scaling_rows]
dfs_nodes = [r["dfs_expanded"] for r in scaling_rows]
astar_nodes = [r["astar_expanded"] for r in scaling_rows]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(counts, bfs_nodes, marker="o", linewidth=2.4, color="#2563EB", label="BFS (Uninformed FIFO)")
ax.plot(counts, dfs_nodes, marker="s", linewidth=2.4, color="#D97706", label="DFS (Uninformed LIFO)")
ax.plot(counts, astar_nodes, marker="^", linewidth=2.4, color="#059669", label="A* Search (Admissible Heuristic)")

ax.set_title("Combinatorial Explosion & Heuristic Tree Pruning (3 to 7 Appliances)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Household Appliances Scheduled", fontsize=11, fontweight="bold")
ax.set_ylabel("Nodes Expanded Before Goal", fontsize=11, fontweight="bold")
ax.set_xticks(counts)
ax.grid(True, linestyle="--", alpha=0.45)
ax.legend(fontsize=10.5)

plt.tight_layout()
plt.show()

## 5. Discussion & Analysis

### Key Findings from Scenario 1 (Proposal 6-Slot Benchmark)

| Metric | BFS | DFS | A* |
|--------|-----|-----|----|
| **Total Cost** | $135.00 | Varies | **$64.00** |
| **Completion Slot** | **Slot 3** (earliest) | Varies | Slot 6 |
| **Optimization Target** | Time | Memory | Cost |

1. **BFS finds the earliest completion** (Slot 3) at a cost of **\$135**, greedily packing maximum load into the first available slots (6u in Slot 1 at \$10/u = \$60, then 5u in Slot 3 at \$15/u = \$75). BFS is time-optimal but cost-oblivious.

2. **A\* finds the global minimum cost of \$64** (a **52.6% savings** vs. BFS), strategically delaying all jobs to the cheapest slots — Slot 4 (\$8/u) and Slot 6 (\$4/u) — completing at Slot 6. The admissible heuristic $h(n) = E_{\text{rem}} \times p_{\min}$ guides A* to prune expensive branches early while guaranteeing optimality.

3. **DFS finds the first feasible schedule** with the **lowest memory footprint** (`max_frontier_size` significantly smaller than BFS). DFS explores depth-first, producing an arbitrary but valid schedule that spreads jobs across multiple slots.

### Heuristic Admissibility & A* Optimality

The heuristic $h(n) = E_{\text{rem}}(s) \times p_{\min}(t)$ is proven admissible because every remaining energy unit must be scheduled at a tariff $\geq p_{\min}(t)$, so $h(s) \leq h^*(s)$ always holds. This guarantees that A* expands nodes in non-decreasing $f$-cost order and the first goal node popped is globally cost-optimal.

### Combinatorial Scaling (Scenario 3)

As the number of appliances increases from 3 to 7, the state space grows combinatorially. The number of valid action subsets per active slot scales as $O(2^N)$, causing **exponential growth in nodes expanded** for uninformed strategies (BFS and DFS). A*'s heuristic-guided pruning dramatically limits the search frontier, expanding far fewer nodes to reach the optimal solution. This empirically confirms the theoretical advantage of informed search in combinatorial optimization domains.

## 6. Concluding Remarks

### Graduate Attribute Assessment

**GA-2 (Problem Analysis):** This project demonstrates rigorous problem decomposition by modeling household appliance scheduling as a formal state-space search problem. The state representation $S = (t, (r_1, \ldots, r_N))$ cleanly encodes the scheduling constraints, while the action generator enforces outage rules and capacity limits. The admissible heuristic $h(n) = E_{\text{rem}} \times p_{\min}$ was formally proven to satisfy $h(s) \leq h^*(s)$, enabling A* to guarantee global cost optimality.

**GA-5 (Modern Tool Usage):** The implementation leverages modern software engineering practices including:
- **Object-Oriented Programming** with frozen dataclasses for immutable, hashable states
- **Interactive Streamlit Dashboard** (`app.py`) for live parameter manipulation during lab demonstrations
- **Jupyter Notebook** (this document) for reproducible, publication-grade academic submission
- **Matplotlib/NumPy** for publication-quality visual analytics (heatmaps, bar charts, scaling plots)
- **Automated pytest suite** for verification of correctness invariants

### Bloom's Taxonomy Alignment

**C3 (Application):** Applied three classical AI search strategies (BFS, DFS, A*) to a real-world constrained scheduling problem, demonstrating how theoretical algorithms behave distinctly when optimizing for different objectives (time vs. cost vs. memory).

**P2 (Mechanism / Guided Response):** Built an interactive mechanism (Streamlit dashboard + Jupyter notebook) enabling guided exploration of how parameter changes (tariffs, outages, capacity) affect search behavior and schedule optimality in real time.

## 7. References

1. Russell, S. J., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. — Chapters 3–4 (Solving Problems by Searching, Search in Complex Environments).

2. CSC-471-L Artificial Intelligence Lab Manual — Lab Session 6: Open-Ended Lab (Iqra University, Department of Computer Science).

3. Python Software Foundation. *Python 3.x Documentation: `dataclasses`, `heapq`, `collections.deque`, `itertools`*. https://docs.python.org/3/

4. Hunter, J. D. (2007). Matplotlib: A 2D Graphics Environment. *Computing in Science & Engineering*, 9(3), 90–95.